In [13]:
#import libraries
import numpy as np
from ase.io import read, write
from ase import Atoms
import torch
import yaml
import json
import matplotlib.pyplot as plt
import pandas as pd
import os
import sys
import re
os.makedirs('img_res', exist_ok=True) #creates a folder to store the loss graphs
os.makedirs('test_res', exist_ok=True) #creates a folder to store the files of the testing of the model
import functions as f #import functions used in this notebook

In [14]:
#variable setup
name = "Fe_Si_B_260311"
type = 'rnd_e'
#setting the model name based on model number and epochs
device = 'cpu'
model = "MACE-matpes-pbe-omat-ft"
model_id = 'matpes_nofe8b4_freeze'
learning_rate = 1e-4
num_epoch = 80 #number of epochs used for training
batch_size = 10 #batch size for training
folder = f'{model_id}_{learning_rate}_{num_epoch}_{batch_size}_{type}'
path = f'model_{name}/freeze_training/{folder}'
train_file = f"model_{name}/train_{type}.xyz"
test_file = f"model_{name}/test_{type}.xyz"
model_name = f'model_{type}_{model_id}_lr{learning_rate}_{num_epoch}_{batch_size}'
os.makedirs(path, exist_ok=True)
os.makedirs(f'{path}/img_res', exist_ok=True)
os.makedirs(f'{path}/test_res', exist_ok=True)
model_name

'model_rnd_e_matpes_nofe8b4_freeze_lr0.0001_80_10'

In [15]:
#setting new names for the training and test files
structure = 'Fe8B4'
train_file = f"model_{name}/train_{type}_no_{structure}.xyz"
test_file = f"model_{name}/test_{type}_no_{structure}.xyz"

In [17]:
train_file

'model_Fe_Si_B_260311/train_rnd_e_no_Fe8B4.xyz'

In [20]:
#this writes the yml file
config = {
    'foundation_model': f'{model}.model',
    'multiheads_finetuning': False,
    'freeze': 6,
    "name": model_name,
    "model_dir": path,
    "log_dir": f"{path}/log",
    "checkpoints_dir": f"{path}/checkpoints",
    "results_dir": f"{path}/results",
    "train_file": train_file,
    "valid_fraction": 0.1,
    "test_file": test_file,
    "energy_key": "REF_energy",
    "forces_key": "REF_forces",
    "batch_size": batch_size,
    "max_num_epochs": num_epoch,
    "lr":learning_rate,
    "device": device,
    "seed": 123
}
with open(f"model_{name}/config_freeze.yml", "w") as f:
    yaml.dump(config, f, sort_keys=False)

In [21]:
#Perform training
import warnings
warnings.filterwarnings('ignore')
from mace.cli.run_train import main as mace_run_train_main
import sys
import logging

#defining the training function
def train_mace(config_file_path):
    logging.getLogger().handlers.clear()
    sys.argv = ['program', '--config', config_file_path]
    mace_run_train_main()

#calling the function
train_mace(f'model_{name}/config_fine_tuning.yml') # use the name of the config file that was created

2026-06-04 14:37:26.412 INFO: ===========VERIFYING SETTINGS===========
2026-06-04 14:37:26.413 INFO: MACE version: 0.3.14
2026-06-04 14:37:26.414 INFO: Using CPU
2026-06-04 14:37:26.579 INFO: Using foundation model MACE-matpes-pbe-omat-ft.model as initial checkpoint.
2026-06-04 14:37:26.580 WARNING: Using multiheads finetuning with a foundation model that is not a Materials Project model, need to provied a path to a pretraining file with --pt_train_file.
2026-06-04 14:37:26.580 INFO: ===========LOADING INPUT DATA===========
2026-06-04 14:37:26.580 INFO: Using heads: ['Default']
2026-06-04 14:37:26.581 INFO: Using the key specifications to parse data:
2026-06-04 14:37:26.581 INFO: Default: KeySpecification(info_keys={'energy': 'REF_energy', 'stress': 'REF_stress', 'virials': 'REF_virials', 'dipole': 'dipole', 'head': 'head', 'elec_temp': 'elec_temp', 'total_charge': 'total_charge', 'polarizability': 'polarizability', 'total_spin': 'total_spin'}, arrays_keys={'forces': 'REF_forces', 'cha